In [ ]:
import pandas as pd
import re
from sqlalchemy import create_engine

In [ ]:
df = pd.read_csv('G55_data.csv', parse_dates=['airline_created_date', 'redemption_date'])
df.head(3)

In [ ]:
df['expiry_days'] = df['reward_comments'].apply(
    lambda c: int(re.search(r'(\d+)\s+days', str(c)).group(1))
)
df['min_miles'] = df['promotion_comments'].apply(
    lambda c: int(re.search(r'over\s+(\d+)\s+miles', str(c)).group(1))
)

In [ ]:
df_reward = df[['reward_id', 'reward_name', 'expiry_days']].copy()
df_reward = df_reward.drop_duplicates(subset='reward_id', keep='first')
df_reward.rename(columns={'reward_name': 'name'}, inplace=True)
df_reward.head(3)

In [ ]:
df_promotion = df[['promotion_id', 'promotion_name', 'min_miles', 'promotion_comments', 'reward_id']].copy()
df_promotion = df_promotion.drop_duplicates(subset='promotion_id', keep='first')
df_promotion.rename(columns={'promotion_name': 'name', 'promotion_comments': 'comments'}, inplace=True)
df_promotion.head(3)

In [ ]:
df_airline = df[['airline_id', 'airline_name', 'airline_created_date']].copy()
df_airline = df_airline.drop_duplicates(subset='airline_id', keep='first')
df_airline.rename(columns={'airline_name': 'name', 'airline_created_date': 'created_date'}, inplace=True)
df_airline['created_date'] = df_airline['created_date'].dt.strftime('%Y-%m-%d')
df_airline.head(3)

In [ ]:
df_link = df[['airline_id', 'promotion_id']].drop_duplicates()
df_link.head(3)

In [ ]:
df_redemption = df[['redemption_id', 'redemption_date', 'passenger_id', 'miles_used', 'promotion_id']].copy()
df_redemption = df_redemption.drop_duplicates(subset='redemption_id', keep='first')
df_redemption['redemption_date'] = df_redemption['redemption_date'].dt.strftime('%Y-%m-%d')
df_redemption.head(3)

In [ ]:
engine = create_engine('sqlite:///G55.db')

df_reward.to_sql('Reward', con=engine, if_exists='replace', index=False)
df_promotion.to_sql('Promotion', con=engine, if_exists='replace', index=False)
df_airline.to_sql('Airline', con=engine, if_exists='replace', index=False)
df_link.to_sql('AirlinePromotion', con=engine, if_exists='replace', index=False)
df_redemption.to_sql('Redemption', con=engine, if_exists='replace', index=False)

In [ ]:
pd.read_sql('SELECT * FROM Reward', con=engine).head(3)

In [ ]:
pd.read_sql('SELECT * FROM Promotion', con=engine).head(3)

In [ ]:
pd.read_sql('SELECT * FROM Airline', con=engine).head(3)

In [ ]:
pd.read_sql('SELECT * FROM AirlinePromotion', con=engine).head(3)

In [ ]:
pd.read_sql('SELECT * FROM Redemption', con=engine).head(3)

In [ ]:
engine.dispose()